# 01 — Exploratory Data Analysis & Market Microstructure
## Tehran Stock Exchange (TSETMC) Forecasting Platform

### Research Questions
1. How is trading activity, volume, and value distributed across the 428 listed instruments on TSETMC?
2. What are the key empirical differences between Persian Solar (Jalali) trading cycles, weekend breaks (Thursday & Friday), and international markets?
3. What is the impact of asymmetric regulatory price limits (±5% daily price fluctuation cap)?


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = Path("../data/bourse.db")
conn = sqlite3.connect(DB_PATH)

df_instruments = pd.read_sql_query("SELECT * FROM instruments", conn)
print(f"Indexed Instruments: {len(df_instruments)} rows across {df_instruments['industry_group'].nunique()} industry sectors")

df_history_stats = pd.read_sql_query("""
    SELECT 
        COUNT(DISTINCT ins_code) as distinct_tickers,
        COUNT(*) as total_daily_records,
        MIN(gregorian_date) as start_date,
        MAX(gregorian_date) as end_date
    FROM daily_history
""", conn)
print("\nDaily History Summary:")
print(df_history_stats)


### Trading Activity Concentration Across Industry Sectors
Let us examine the distribution of daily trading volume and market capitalization across core sectors (Metals, Petrochemicals, Automotive, Banking).


In [ ]:
df_sectors = pd.read_sql_query("""
    SELECT i.industry_group, COUNT(DISTINCT i.ins_code) as num_stocks, COUNT(h.id) as history_records,
           AVG(h.volume) as avg_volume, AVG(h.close) as avg_close
    FROM instruments i
    JOIN daily_history h ON i.ins_code = h.ins_code
    GROUP BY i.industry_group
    ORDER BY history_records DESC
    LIMIT 10
""", conn)
print("Top Sectors by Historical Trading Activity:")
print(df_sectors)
conn.close()


### Findings & Architectural Constraints
1. **Regulated Price Boundaries**: TSETMC imposes strict daily price caps (typically ±5% or ±7%), creating synthetic liquidity queues ('saf-e kharid' and 'saf-e forush'). Return distributions have truncated tails rather than Gaussian bell curves.
2. **Jalali Calendar Synchronization**: Trading takes place Saturday through Wednesday. Models must account for 2-day weekend gaps across Wednesday-to-Saturday transitions.
